# Voxel atlas and stat

Port of `packages/niivue/examples/vox.atlas.stat.html`. Combines an anatomical template, the AAL atlas label map, and a statistical overlay.

This notebook also exercises the JS-to-Python event path: `nv.on("locationChange", ...)` updates a footer label on every crosshair move, mirroring the HTML demo's `<footer id="location">` behavior. Drag the crosshair around for a sustained period to load-test the `_msg_outbox` channel.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"


def hex_to_rgba(value):
    value = value.lstrip("#")
    return [int(value[i : i + 2], 16) / 255 for i in (0, 2, 4)] + [1.0]


nv = NiiVue(
    background_color=[0.3, 0.3, 0.5, 1.0],
    is_colorbar_visible=True,
    show_render=1,
    slice_type=3,
    backend="webgl2",
)

slice_type = widgets.Dropdown(
    options=[("Axial", 0), ("Coronal", 1), ("Sagittal", 2), ("A+C+S+R", 3), ("Render", 4)],
    value=3,
    description="Slice",
)
atlas_opacity = widgets.IntSlider(value=50, min=1, max=100, step=1, description="Atlas")
stat_opacity = widgets.IntSlider(value=50, min=1, max=100, step=1, description="Stat")
nearest = widgets.Checkbox(value=True, description="Nearest")
clip_dark = widgets.Checkbox(value=True, description="Clip dark")
legend = widgets.Checkbox(value=False, description="Legend")
colorbar = widgets.Checkbox(value=True, description="Colorbars")
background = widgets.ColorPicker(value="#4d4d80", description="Background")

# Footer label, fed by the JS-side locationChange event. The HTML demo
# writes detail.string into <footer id="location">; we do the same.
location_label = widgets.Label(value="NiiVue")


def update_slice(change):
    nv.slice_type = change["new"]


def update_atlas_opacity(change):
    nv.set_volume(1, {"opacity": change["new"] * 0.01})


def update_stat_opacity(change):
    nv.set_volume(2, {"opacity": change["new"] * 0.01})


def update_nearest(change):
    nv.volume_is_nearest_interpolation = change["new"]


def update_clip_dark(change):
    nv.volume_is_alpha_clip_dark = change["new"]


def update_legend(change):
    nv.set_volume(1, {"isLegendVisible": change["new"]})


def update_colorbar(change):
    nv.is_colorbar_visible = change["new"]


def update_background(change):
    nv.background_color = hex_to_rgba(change["new"])


def on_location_change(detail):
    # detail comes from NiiVue's locationChange CustomEvent; .string is
    # the pre-formatted "x=... y=... z=... intensity=..." line.
    location_label.value = (detail or {}).get("string", "") or ""


slice_type.observe(update_slice, names="value")
atlas_opacity.observe(update_atlas_opacity, names="value")
stat_opacity.observe(update_stat_opacity, names="value")
nearest.observe(update_nearest, names="value")
clip_dark.observe(update_clip_dark, names="value")
legend.observe(update_legend, names="value")
colorbar.observe(update_colorbar, names="value")
background.observe(update_background, names="value")
nv.on("locationChange", on_location_change)

controls = widgets.VBox(
    [
        widgets.HBox([slice_type, atlas_opacity, stat_opacity, background]),
        widgets.HBox([nearest, clip_dark, legend, colorbar]),
    ]
)
display(controls)
display(nv)
display(location_label)

nv.load_volumes(
    [
        {"url": f"{VOLUMES}/mni152.nii.gz", "calMin": 30, "calMax": 80, "isColorbarVisible": False},
        {"url": f"{VOLUMES}/aal.nii.gz"},
        {"url": f"{VOLUMES}/spmMotor.nii.gz", "colormap": "hot", "calMin": 3, "calMax": 8},
    ]
)
nv.set_colormap_label_from_url(1, f"{VOLUMES}/aal.json")
update_nearest({"new": nearest.value})
update_clip_dark({"new": clip_dark.value})
update_atlas_opacity({"new": atlas_opacity.value})
update_stat_opacity({"new": stat_opacity.value})